In [ ]:
#@title Install required libraries
!pip install -q anthropic


In [ ]:
#@title Import libraries and setup
import os
import json
from anthropic import Anthropic

# Configure API key: Colab userdata first, then environment variable
api_key = None
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    raise RuntimeError(
        "Missing ANTHROPIC_API_KEY. Set it in Colab userdata or as an environment variable."
    )

# Initialize Anthropic client with prompt caching enabled
client = Anthropic(
    api_key=api_key,
    default_headers={"anthropic-beta": "prompt-caching-2024-07-31"}
)

# Claude Haiku model name (adjust if a newer 4.5 Haiku identifier is available)
MODEL_NAME = os.environ.get("CLAUDE_MODEL", "claude-haiku-4-5-20251001")


In [ ]:
#@title Data Loading Functions
def load_kg_data(file_path):
    """Load Knowledge Graph JSON file as raw string"""
    try:
        with open(file_path, 'r') as f:
            return json.dumps(json.load(f)), None
    except Exception as e:
        return None, f"Error loading KG file: {str(e)}"


def load_raw_data(file_path):
    """Load raw text file as string"""
    try:
        with open(file_path, 'r') as f:
            return f.read(), None
    except Exception as e:
        return None, f"Error loading raw file: {str(e)}"

In [ ]:
#@title Prompt Templates
# Static parts (cacheable) - instructions that don't change
KG_PROMPT_STATIC = """
1. Focus on the question:
Analyze the provided TraceCompass State System knowledge graph data to answer the question. You must focus on answering this specific question, and your response should be derived exclusively from the provided knowledge graph data.

2. Use the provided data:
The data you will use is from the TraceCompass State System, specifically in the form of a knowledge graph. Ensure your analysis strictly uses this data for the answer.

3. Leverage your knowledge:
You may use your understanding of TraceCompass State System analysis methods (such as CPU usage analysis, active thread analysis, and Ease script queries) as context for interpreting the data. Keep in mind that the provided data comes from an Ease script query run on the TraceCompass system.

4. Reason over the graph:
The data may not always be explicitly present in the graph, and you may need to reason over the graph's structure and relationships to derive the answer. Use your knowledge of node relationships, temporal patterns, and resource utilization to infer the missing information where necessary.

5. Answer directly:
Provide a direct, concise answer to the question. This answer should be based solely on the provided data.

6. Explain your reasoning:
After providing the direct answer, explain how you arrived at it. Your explanation should cover the following:
   - How the nodes and their relationships helped answer the question.
   - Any specific behaviors or insights derived from the TraceCompass State System analysis, such as CPU usage or thread activity analysis.

7. Question and context:
"""

RAW_PROMPT_STATIC = """
1. Focus on the question:
Analyze the provided raw TraceCompass system trace data to answer the question. Your response must be directly based on this data.

2. Use the provided data:
The data you will use is from the TraceCompass State System in the form of raw system trace measurements. Your analysis should rely exclusively on this data for your answer.

3. Leverage your knowledge:
You may use your understanding of TraceCompass State System analysis methods (such as CPU usage analysis, active thread analysis, and Ease script queries) to interpret the trace data. The provided data is the result of an Ease script query run on the TraceCompass system.

4. Answer directly:
Provide a direct, concise answer to the question based on the raw trace data.

5. Explain your reasoning:
After providing the direct answer, explain how the raw data supports your answer. In your explanation, you should cover:
   - Any patterns or anomalies you identified in the trace data.
   - How these patterns or anomalies are relevant to the TraceCompass system's performance or behavior, particularly related to CPU usage, thread activity, or any Ease script queries.
   - Any performance characteristics or insights drawn from the data, emphasizing the specific context of TraceCompass' analysis.

6. Question and context:
"""


In [ ]:
#@title Analysis Functions
def query_claude(static_prompt: str, context_data: str, question: str, context_label: str = "TraceCompass Knowledge Graph Context:", cache_id: str = "default") -> str:
    """Execute Claude Haiku query using Anthropic Messages API with prompt caching.
    
    Args:
        static_prompt: The static part of the prompt (cacheable instructions)
        context_data: The context data (cacheable, e.g., JSON or raw data)
        question: The question (dynamic, not cached)
        context_label: Label for the context (e.g., "TraceCompass Knowledge Graph Context:" or "Raw Data Context:")
        cache_id: Unique identifier for the cache (default: "default")
    """
    try:
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=1200,
            temperature=0.5,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": static_prompt,
                            "cache_control": {"type": "ephemeral"}  # Cache the static prompt
                        }
                    ]
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"{context_label}\n{context_data}",
                            "cache_control": {"type": "ephemeral"}  # Cache the context data
                        }
                    ]
                },
                {
                    "role": "user",
                    "content": f"Question: {question}"  # Question is not cached (changes each time)
                }
            ]
        )
        # Extract text content from first block if present
        if getattr(response, "content", None):
            for block in response.content:
                if getattr(block, "type", None) == "text" and getattr(block, "text", None):
                    return block.text.strip()
        # Fallback to string repr
        return str(response).strip()
    except Exception as e:
        return f"API Error: {str(e)}"


def analyze_kg(question: str, file_path: str) -> str:
    """Knowledge Graph analysis pipeline with prompt caching"""
    context, error = load_kg_data(file_path)
    if error:
        return error
    
    # Pass context and question separately - context will be cached
    return query_claude(
        static_prompt=KG_PROMPT_STATIC,
        context_data=context,
        question=question,
        cache_id=f"kg_{file_path}"  # Unique cache ID based on file path
    )


def analyze_raw(question: str, file_path: str) -> str:
    """Raw data analysis pipeline with prompt caching"""
    context, error = load_raw_data(file_path)
    if error:
        return error
    
    # Pass context and question separately - context will be cached
    return query_claude(
        static_prompt=RAW_PROMPT_STATIC,
        context_data=context,
        question=question,
        context_label="Raw Data Context:",
        cache_id=f"raw_{file_path}"  # Unique cache ID based on file path
    )


In [ ]:
#@title Main Execution
# def full_analysis(question: str, kg_path: str = "data/cpu_usage_graph.json", raw_path: str = "data/cpu_usage_input.txt"):
#     """Run complete analysis with both approaches"""
#     print(f"\n🔍 Question: {question}")
#     print("="*60)
# 
#     print("\n\n📈 Raw Data Analysis:")
#     print(analyze_raw(question, raw_path))
# 
#     print("\n📊 Knowledge Graph Analysis:")
#     print(analyze_kg(question, kg_path))


# Example usage: test only analyze_kg
if __name__ == "__main__":
    sample_question = "What is the total accumulated CPU time for thread 5130 on CPU 2?"  #@param {type:"string"}
    kg_path = "data/cpu_usage_graph.json"
    print(f"\n🔍 Question: {sample_question}")
    print("="*60)
    print("\n📊 Knowledge Graph Analysis:")
    print(analyze_kg(sample_question, kg_path))
